In [ ]:
!pip install -q requests pymupdf
!pip install sentence-transformers hdbscan scikit-learn pandas numpy matplotlib

In [ ]:
import fitz  # PyMuPDF
import requests
import json

In [ ]:
OLLAMA_API_URL = "https://api.ollama.com/api/generate"
# If Ollama Cloud gives you another URL, replace it.

OLLAMA_API_KEY = "28de8464911240dc9b29591a0c97bf12.XcaV7bc7kZnrTnqwfMHuxR66"

MODEL_NAME = "gemma3:4b-cloud"

In [ ]:
def extract_text_from_pdf(pdf_path, max_pages=8):
    doc = fitz.open(pdf_path)
    text = ""

    for i in range(min(max_pages, len(doc))):
        page = doc.load_page(i)
        text += page.get_text()

    return text.strip()

In [ ]:
def classify_publication_auto(title, abstract_text, pdf_text):

    prompt = f"""
You are an expert scientific publication classifier.

Your task is to analyze the following scientific publication and automatically determine the most relevant scientific categories/domains.

### INPUT DATA
Title:
{title}

Abstract:
{abstract_text}

Extracted PDF Content:
{pdf_text[:4000]}

### TASK
1. Determine the most relevant scientific categories (domains).
2. Provide the top 5 categories (most relevant first).
3. Provide a confidence score between 0 and 1.
4. Provide a short justification for each category.
5. Also generate 5 keywords describing the paper.

### OUTPUT FORMAT
Return ONLY valid JSON in the following format:

{{
  "title": "...",
  "predicted_categories": [
    {{
      "category": "...",
      "confidence": 0.0,
      "reason": "..."
    }}
  ],
  "keywords": ["...", "", "", "", ""]
}}
"""

    payload = {
        "model": MODEL_NAME,
        "prompt": prompt,
        "stream": False
    }

    headers = {
        "Authorization": f"Bearer {OLLAMA_API_KEY}",
        "Content-Type": "application/json"
    }

    response = requests.post(OLLAMA_API_URL, json=payload, headers=headers)

    if response.status_code != 200:
        raise Exception("Ollama API Error: " + response.text)

    result = response.json()
    return result["response"]

In [ ]:
data_simple = {
      "title" : "Learning to Grasp in Clutter Using Contact Mechanics",
      "abstract_text" : "We present a learning-based approach to grasping objects in cluttered environments. Our method uses contact mechanics to reason about stable grasps and applies reinforcement learning to acquire a policy that can generalize to novel object geometries and arrangements.",
      "pdf_url" : "https://arxiv.org/pdf/2010.12321.pdf"
}

title = data_simple["title"]
abstract_text = data_simple["abstract_text"]
pdf_url = data_simple["pdf_url"]

In [ ]:
from google.colab import files

response = requests.get(pdf_url)
response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)

pdf_filename = pdf_url.split('/')[-1] # Extract filename from URL
with open(pdf_filename, 'wb') as f:
    f.write(response.content)

print(f"Downloaded PDF to: {pdf_filename}")

In [ ]:
pdf_text = extract_text_from_pdf(pdf_filename, max_pages=8)

print("PDF text extracted successfully!")
print("Characters extracted:", len(pdf_text))

In [ ]:
result_json = classify_publication_auto(title, abstract_text, pdf_text)

print("===== RAW MODEL OUTPUT =====")
print(result_json)

Scientific Publication Clustering (MiniLM + HDBSCAN)

In [ ]:
import fitz  # PyMuPDF
import requests
import pandas as pd
import numpy as np
import hdbscan
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
def download_pdf(pdf_url):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    response = requests.get(pdf_url, stream=True, headers=headers)

    if response.status_code != 200:
        raise Exception(f"Failed to download PDF. Status code: {response.status_code}")

    save_path = pdf_url.split('/')[-1] # Extract filename from URL
    with open(save_path, "wb") as f:
        for chunk in response.iter_content(chunk_size=1024):
            f.write(chunk)

    return save_path

In [ ]:
def extract_text_from_pdf(pdf_path, max_pages=8):
    doc = fitz.open(pdf_path)
    text = ""

    for i in range(min(max_pages, len(doc))):
        page = doc.load_page(i)
        text += page.get_text()

    return text.strip()

In [ ]:
def build_publication_text(abstract, pdf_text, domain=None):
    full_text = ""

    if domain and domain.strip() != "":
        full_text += f"Domain: {domain}\n\n"

    full_text += f"Abstract:\n{abstract}\n\n"
    full_text += f"PDF Content:\n{pdf_text[:4000]}"

    return full_text.strip()

In [ ]:
publications = [
    # {
    #     "pdf_url": "https://ieeexplore.ieee.org/document/1638022",
    #     "abstract": "This paper discusses the simultaneous localization and mapping (SLAM) problem, in which a mobile robot builds a map of an unknown environment while simultaneously navigating within that environment. This is a fundamental problem in mobile robotics.",
    #     "domain": "Robotics"
    # },
    # {
    #     "pdf_url" : "https://arxiv.org/pdf/2010.12321.pdf",
    #     "abstract" : "We present a learning-based approach to grasping objects in cluttered environments. Our method uses contact mechanics to reason about stable grasps and applies reinforcement learning to acquire a policy that can generalize to novel object geometries and arrangements.",
    # },
    {
        "abstract": 'We present YOLO, a new approach to object detection. Prior work repurposes classifiers to perform detection. Instead, we frame object detection as a regression problem to spatially separated bounding boxes and associated class probabilities.',
        "pdf_url": 'https://arxiv.org/pdf/1506.02640.pdf',
        "domain": 'Computer Vision'
    },
    {
        "domain": 'Computer Vision',
        "abstract": 'We introduce a Region Proposal Network (RPN) that shares full-image convolutional features with the detection network, enabling nearly cost-free region proposals. An RPN is a fully convolutional network that simultaneously predicts object bounds and objectness scores at each position.',
        "pdf_url": 'https://arxiv.org/pdf/1506.01497.pdf',
    },
    {
        "domain": 'Computer Vision',
        "abstract": 'We present a method for detecting objects in images using a single deep neural network. Our approach, named SSD, discretizes the output space of bounding boxes into a set of default boxes over different aspect ratios and scales per feature map location.',
        "pdf_url": 'https://arxiv.org/pdf/1512.02325.pdf'
    },
    {
        "domain": 'Computer Vision',
        "abstract": 'There are a huge number of features which are said to improve Convolutional Neural Network accuracy. Practical testing of combinations of such features on large datasets requires significant GPU time. YOLOv4 achieves a new state-of-the-art balance between accuracy and speed.',
        "pdf_url": 'https://arxiv.org/pdf/2004.10934.pdf'
    },
    {
        "domain": 'Computer Vision',
        "abstract": 'There is large consent that successful training of deep networks requires many thousand annotated training samples. In this paper, we present a network and training strategy that relies on the strong use of data augmentation to use the available annotated samples more efficiently.',
        "pdf_url": 'https://arxiv.org/pdf/1411.4038.pdf'
    },
    {
        "domain": 'Computer Vision',
        "abstract": 'Convolutional networks are powerful visual models that yield hierarchies of features. We show that convolutional networks by themselves, trained end-to-end and pixels-to-pixels, exceed the state-of-the-art in semantic segmentation.',
        "pdf_url": 'https://arxiv.org/pdf/1505.04597.pdf'
    },
    {
        "domain": 'Computer Vision',
        "abstract": 'We present a conceptually simple, flexible, and general framework for object instance segmentation. Our approach efficiently detects objects in an image while simultaneously generating a high-quality segmentation mask for each instance.',
        "pdf_url": 'https://arxiv.org/pdf/1703.06870.pdf'
    },

]

In [ ]:
dataset = []

for i, pub in enumerate(publications):
    print(f"\nProcessing publication {i+1}/{len(publications)}")

    pdf_path = download_pdf(pub["pdf_url"])
    pdf_text = extract_text_from_pdf(pdf_path, max_pages=8)

    full_text = build_publication_text(
        abstract=pub["abstract"],
        pdf_text=pdf_text,
        domain=pub.get("domain", "")
    )

    dataset.append({
        "pdf_url": pub["pdf_url"],
        "abstract": pub["abstract"],
        "domain": pub.get("domain", ""),
        "full_text": full_text
    })

df = pd.DataFrame(dataset)
df.head()

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

texts = df["full_text"].tolist()

embeddings = model.encode(texts, show_progress_bar=True)
embeddings = np.array(embeddings)

print("Embeddings shape:", embeddings.shape)

In [ ]:
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=2,
    min_samples=1,
    metric="euclidean"
)

cluster_labels = clusterer.fit_predict(embeddings)

df["cluster"] = cluster_labels
df[["pdf_url", "cluster"]]

In [ ]:
def extract_cluster_keywords(df, cluster_id, top_n=8):
    cluster_texts = df[df["cluster"] == cluster_id]["full_text"].tolist()

    if len(cluster_texts) == 0:
        return []

    vectorizer = TfidfVectorizer(stop_words="english", max_features=2000)
    X = vectorizer.fit_transform(cluster_texts)

    scores = np.asarray(X.sum(axis=0)).flatten()
    terms = np.array(vectorizer.get_feature_names_out())

    top_indices = scores.argsort()[::-1][:top_n]
    return terms[top_indices].tolist()

In [ ]:
clusters = sorted(df["cluster"].unique())

for c in clusters:
    if c == -1:
        print("\n==============================")
        print("NOISE / OUTLIERS (-1)")
        print("==============================")
        print(df[df["cluster"] == -1]["pdf_url"].tolist())
        continue

    keywords = extract_cluster_keywords(df, c)

    print("\n==============================")
    print(f"CLUSTER {c}")
    print("==============================")
    print("Keywords:", keywords)
    print("Publications URLs:", df[df["cluster"] == c]["pdf_url"].tolist())

In [ ]:
def compute_cluster_centroids(df, embeddings):
    centroids = {}
    for cluster_id in df["cluster"].unique():
        if cluster_id == -1:
            continue
        indices = df[df["cluster"] == cluster_id].index.tolist()
        centroids[cluster_id] = np.mean(embeddings[indices], axis=0)
    return centroids

centroids = compute_cluster_centroids(df, embeddings)

In [ ]:
def predict_cluster_for_new_publication(pdf_url, abstract, domain=None):
    pdf_path = download_pdf(pdf_url)
    pdf_text = extract_text_from_pdf(pdf_path, max_pages=8)

    full_text = build_publication_text(abstract, pdf_text, domain)

    new_embedding = model.encode([full_text])
    new_embedding = np.array(new_embedding)

    best_cluster = None
    best_score = -1

    for cluster_id, centroid in centroids.items():
        score = cosine_similarity(new_embedding, centroid.reshape(1, -1))[0][0]
        if score > best_score:
            best_score = score
            best_cluster = cluster_id

    return best_cluster, best_score

In [ ]:
new_pdf_url = 'https://arxiv.org/pdf/2105.15203.pdf'
new_abstract =   'We present SegFormer, a simple, efficient yet powerful semantic segmentation framework which unifies Transformers with lightweight multilayer perceptron decoders. SegFormer has two appealing features: a novel hierarchically structured Transformer encoder that outputs multiscale features.'
new_domain = 'Computer Vision'

cluster_id, similarity_score = predict_cluster_for_new_publication(
    new_pdf_url,
    new_abstract,
    new_domain
)

print("Predicted Cluster:", cluster_id)
print("Similarity Score:", similarity_score)

Recommend similar publications

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def recommend_similar_publications(df, embeddings, target_index, top_k=5):
    """
    Recommend similar publications to a publication already in dataset.
    """

    target_embedding = embeddings[target_index].reshape(1, -1)

    sims = cosine_similarity(target_embedding, embeddings)[0]

    # sort indices by similarity descending
    similar_indices = sims.argsort()[::-1]

    # remove itself
    similar_indices = [idx for idx in similar_indices if idx != target_index]

    top_indices = similar_indices[:top_k]

    results = df.iloc[top_indices].copy()
    results["similarity_score"] = sims[top_indices]

    return results[["pdf_url", "domain", "cluster", "similarity_score"]]

In [ ]:
target_index = 0  # choose a paper in your dataset
recommendations = recommend_similar_publications(df, embeddings, target_index, top_k=5)

print("Recommendations for:", df.loc[target_index, "pdf_url"])
recommendations

Recommend ONLY from the same cluster

In [ ]:
def recommend_with_cluster_filter(df, embeddings, target_index, top_k=5):
    target_cluster = df.loc[target_index, "cluster"]

    if target_cluster == -1:
        print("This paper is noise/outlier, cluster-based recommendation not possible.")
        return None

    cluster_indices = df[df["cluster"] == target_cluster].index.tolist()

    target_embedding = embeddings[target_index].reshape(1, -1)
    cluster_embeddings = embeddings[cluster_indices]

    sims = cosine_similarity(target_embedding, cluster_embeddings)[0]

    sorted_idx = sims.argsort()[::-1]

    sorted_idx = [idx for idx in sorted_idx if cluster_indices[idx] != target_index]

    top_local = sorted_idx[:top_k]
    top_indices = [cluster_indices[i] for i in top_local]

    results = df.loc[top_indices].copy()
    results["similarity_score"] = sims[top_local]

    return results[["pdf_url", "domain", "cluster", "similarity_score"]]

In [ ]:
recommend_with_cluster_filter(df, embeddings, target_index=0, top_k=5)

Detect Close Papers

In [ ]:
def detect_close_pairs(df, embeddings, threshold=0.90):
    """
    Detect pairs of publications with similarity above a threshold.
    """

    sim_matrix = cosine_similarity(embeddings)

    close_pairs = []

    n = len(df)

    for i in range(n):
        for j in range(i+1, n):
            if sim_matrix[i, j] >= threshold:
                close_pairs.append({
                    "paper_1": df.loc[i, "pdf_url"],
                    "paper_2": df.loc[j, "pdf_url"],
                    "similarity_score": sim_matrix[i, j]
                })

    return pd.DataFrame(close_pairs)

In [ ]:
close_df = detect_close_pairs(df, embeddings, threshold=0.75)

print("Close / duplicate publications found:", len(close_df))
close_df

Recommend Similar Publications for a NEW Publication

In [ ]:
def recommend_for_new_publication(df, embeddings, model, pdf_url, abstract, domain=None, top_k=5):
    pdf_path = download_pdf(pdf_url)
    pdf_text = extract_text_from_pdf(pdf_path, max_pages=8)

    full_text = build_publication_text(abstract, pdf_text, domain)

    new_embedding = model.encode([full_text])
    new_embedding = np.array(new_embedding)

    sims = cosine_similarity(new_embedding, embeddings)[0]

    top_indices = sims.argsort()[::-1][:top_k]

    results = df.iloc[top_indices].copy()
    results["similarity_score"] = sims[top_indices]

    return results[["pdf_url", "domain", "cluster", "similarity_score"]]

In [ ]:
new_pdf_url = 'https://arxiv.org/pdf/2105.15203.pdf'
new_abstract =   'We present SegFormer, a simple, efficient yet powerful semantic segmentation framework which unifies Transformers with lightweight multilayer perceptron decoders. SegFormer has two appealing features: a novel hierarchically structured Transformer encoder that outputs multiscale features.'
new_domain = 'Computer Vision'

recommendations_new = recommend_for_new_publication(
    df, embeddings, model,
    pdf_url=new_pdf_url,
    abstract=new_abstract,
    domain=new_domain,
    top_k=5
)

recommendations_new